In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from scipy.stats import randint
from xgboost import XGBClassifier



In [3]:
df = pd.read_csv("../data/processed/customer_clusters.csv")
# print(df.columns)

In [4]:
X = df.drop("Churn",axis=1)
y = df["Churn"].map({
    "No":0,
    "Yes":1
})

X_train , X_test ,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
numeric = X_train.select_dtypes(
    include = ["int64","float64"]
).columns.tolist()

# print(numeric)


categorique = X_train.select_dtypes(
    include=["object"]
).columns.tolist()
# print(categorique)

prepocessor = ColumnTransformer(
    transformers=[(
            "num",StandardScaler(),numeric
        ),
        (
            "cat",OneHotEncoder(handle_unknown="ignore"),categorique
        )
    ]
)
X_train_prepared = prepocessor.fit_transform(X_train)

X_test_prepared = prepocessor.transform(X_test)


In [6]:
smote= SMOTE(random_state=42)

X_train_smote,y_train_smote = smote.fit_resample(X_train_prepared,y_train)

print(y_train.value_counts(normalize=True))
print(y_train_smote.value_counts(normalize=True))

Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Churn
0    0.5
1    0.5
Name: proportion, dtype: float64


### logostique regresion 

In [7]:
logistic_model = LogisticRegression(
    random_state=42
)

logistic_model.fit(
    X_train_smote,
    y_train_smote
)
y_pre_logistic = logistic_model.predict(X_test_prepared)

precision_logistic_clustring = precision_score(y_test,y_pre_logistic)
recall_logistic_clustring = recall_score(y_test,y_pre_logistic)
f1_logistic_clustring = f1_score(y_test,y_pre_logistic)
roc_auc_logistic_clustring = roc_auc_score(y_test,y_pre_logistic)
confusion_matrix_logistic_clustring = confusion_matrix(y_test,y_pre_logistic)

# params 
# c =>force de regularisation
# penalty => typede regularisation 
# solver 
# max_iter => total d'iteration
# class_weight => pour equilibre des class 

In [8]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"],
    "max_iter": [1000]
}


random_search = RandomizedSearchCV(
    estimator=logistic_model,
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

random_search.fit(
    X_train_smote,
    y_train_smote
)

print("Best parameters:")
print(random_search.best_params_)
best_model = random_search.best_estimator_

y_pre_logistic_optimise = best_model.predict(X_test_prepared)


precision_logistic_clustring_opt = precision_score(y_test,y_pre_logistic_optimise)
recall_logistic_clustring_opt = recall_score(y_test,y_pre_logistic_optimise)
f1_logistic_clustring_opt = f1_score(y_test,y_pre_logistic_optimise)
roc_auc_logistic_clustring_opt = roc_auc_score(y_test,y_pre_logistic_optimise)
confusion_matrix_logistic_clustring_opt = confusion_matrix(y_test,y_pre_logistic_optimise)




# print("precision : ",precision_logistic_opt)
# print("recall : ",recall_logistic_opt)
# print("f1 score : ",f1_logistic_opt)
# print("roc_auc : ",roc_auc_logistic_opt)
# print("confusion matrix : ",confusion_matrix_logistic_opt)



Best parameters:
{'solver': 'liblinear', 'penalty': 'l1', 'max_iter': 1000, 'C': 1}


### decision three


In [9]:
dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(X_train_smote,y_train_smote)

dt_predict = dt.predict(X_test_prepared)

precision_dt_clustring = precision_score(y_test,dt_predict)
recall_dt_clustring = recall_score(y_test,dt_predict)
f1_dt_clustring = f1_score(y_test,dt_predict)
roc_auc_dt_clustring = roc_auc_score(y_test,dt_predict)
confusion_matrix_dt_clustring = confusion_matrix(y_test,dt_predict)


In [10]:
param_grid = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": randint(3, 20),
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 10),
    "max_features": [None, "sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

random_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=50,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

random_search.fit(
    X_train_smote,
    y_train_smote
)


best_model_dt = random_search.best_estimator_

dt_predict_opt = best_model_dt.predict(X_test_prepared)

dt_proba_opt = best_model_dt.predict_proba(X_test_prepared)[:, 1]

precision_dt_clustring_opt = precision_score(y_test,dt_predict_opt)
recall_dt_clustring_opt = recall_score(y_test,dt_predict_opt)
f1_dt_clustring_opt = f1_score(y_test,dt_predict_opt)
roc_auc_dt_clustring_opt = roc_auc_score(y_test,dt_proba_opt)
confusion_matrix_dt_clustring_opt = confusion_matrix(y_test,dt_predict_opt)


### Random Forest

In [29]:
random_model = RandomForestClassifier(
    random_state=42
)

random_model.fit(
    X_train_smote,
    y_train_smote
)

y_pred_rf = random_model.predict(X_test_prepared)


precision_random_clustring = precision_score(y_test,y_pred_rf)
recall_random_clustring = recall_score(y_test,y_pred_rf)
f1_random_clustring = f1_score(y_test,y_pred_rf)
roc_auc_random_clustring = roc_auc_score(y_test,y_pred_rf)
confusion_matrix_random_clustring = confusion_matrix(y_test,y_pred_rf)


In [30]:
param_dist = {
    "n_estimators": [100, 200, 300, 500, 800],
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None]
}

random_search = RandomizedSearchCV(
    estimator=random_model,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_smote, y_train_smote)

best_model_random = random_search.best_estimator_
y_pre_random_opt = best_model.predict(X_test_prepared)



precision_random_clustring_opt = precision_score(y_test,y_pre_random_opt)
recall_random_clustring_opt = recall_score(y_test,y_pre_random_opt)
f1_random_clustring_opt = f1_score(y_test,y_pre_random_opt)
roc_auc_random_clustring_opt = roc_auc_score(y_test,y_pre_random_opt)
confusion_matrix_random_clustring_opt = confusion_matrix(y_test,y_pre_random_opt)


Fitting 5 folds for each of 30 candidates, totalling 150 fits


XGBoost


In [31]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb.fit(X_train_smote,y_train_smote)

y_pred_xgb = xgb.predict(X_test_prepared)

precision_xgboost_clustring = precision_score(y_test,y_pred_xgb)
recall_xgboost_clustring = recall_score(y_test,y_pred_xgb)
f1_xgboost_clustring = f1_score(y_test,y_pred_xgb)
roc_auc_xgboost_clustring = roc_auc_score(y_test,y_pred_xgb)
confusion_matrix_xgboost_clustring = confusion_matrix(y_test,y_pred_xgb)

In [32]:
param_dist_xgb = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [2, 3, 4, 5, 6],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.2, 0.5]
}

random_search_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=30,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search_xgb.fit(X_train_smote, y_train_smote)

y_pred_xgb_opt = random_search_xgb.predict(X_test_prepared)

precision_xgb_clustring_opt = precision_score(y_test,y_pred_xgb_opt)
recall_xgb_clustring_opt = recall_score(y_test,y_pred_xgb_opt)
f1_xgb_clustring_opt = f1_score(y_test,y_pred_xgb_opt)
roc_auc_xgb_clustring_opt = roc_auc_score(y_test,y_pred_xgb_opt)
confusion_matrix_xgb_clustring_opt = confusion_matrix(y_test,y_pred_xgb_opt)


Fitting 5 folds for each of 30 candidates, totalling 150 fits


In [35]:
evaluation = pd.DataFrame({
    "Model": [
        "Logistic Regression clustring",
        "Logistic Regression clustring opt",
        "Random Foreast clustring",
        "Random Foreast clustring opt",
        "Decision three clustring",
        "Decision three clustring opt",
        "Xgboost clustring",
        "Xgboost clustring opt",
    ],
    "Precision": [
        precision_logistic_clustring,
        precision_logistic_clustring_opt,
        precision_random_clustring,
        precision_random_clustring_opt,
        precision_dt_clustring,
        precision_dt_clustring_opt,
        precision_xgboost_clustring,
        precision_xgb_clustring_opt,
    ],
    "Recall": [
        recall_logistic_clustring,
        recall_logistic_clustring_opt,
        recall_random_clustring,
        recall_random_clustring_opt,
        recall_dt_clustring,
        recall_dt_clustring_opt,
        recall_xgboost_clustring,
        recall_xgb_clustring_opt,
    ],
    "F1-score": [
        f1_logistic_clustring,
        f1_logistic_clustring_opt,
        f1_random_clustring,
        f1_random_clustring_opt,
        f1_dt_clustring,
        f1_dt_clustring_opt,
        f1_xgboost_clustring,
        f1_xgb_clustring_opt,
    ],
    "ROC-AUC": [
        roc_auc_logistic_clustring,
        roc_auc_logistic_clustring_opt,
        roc_auc_random_clustring,
        roc_auc_random_clustring_opt,
        roc_auc_dt_clustring,
        roc_auc_dt_clustring_opt,
        roc_auc_xgboost_clustring,
        roc_auc_xgb_clustring_opt,
    ],
    "TN": [
        confusion_matrix_logistic_clustring[0, 0],
        confusion_matrix_logistic_clustring_opt[0, 0],
        confusion_matrix_random_clustring[0,0],
        confusion_matrix_random_clustring_opt[0,0],
        confusion_matrix_dt_clustring[0,0],
        confusion_matrix_dt_clustring_opt[0,0],
        confusion_matrix_xgboost_clustring[0,0],
        confusion_matrix_xgb_clustring_opt[0,0],
    ],

    "FP": [
        confusion_matrix_logistic_clustring[0, 1],
        confusion_matrix_logistic_clustring_opt[0, 1],
        confusion_matrix_random_clustring[0,1],
        confusion_matrix_random_clustring_opt[0,1],
        confusion_matrix_dt_clustring[0,1],
        confusion_matrix_dt_clustring_opt[0,1],
        confusion_matrix_xgboost_clustring[0,1],
        confusion_matrix_xgb_clustring_opt[0,1],
    ],

    "FN": [
        confusion_matrix_logistic_clustring[1, 0],
        confusion_matrix_logistic_clustring_opt[1, 0],
        confusion_matrix_random_clustring[1,0],
        confusion_matrix_random_clustring_opt[1,0],
        confusion_matrix_dt_clustring[1,0],
        confusion_matrix_dt_clustring_opt[1,0],
        confusion_matrix_xgboost_clustring[1,0],
        confusion_matrix_xgb_clustring_opt[1,0],
    ],

    "TP": [
        confusion_matrix_logistic_clustring[1, 1],
        confusion_matrix_logistic_clustring_opt[1, 1],
        confusion_matrix_random_clustring[1,1],
        confusion_matrix_random_clustring_opt[1,1],
        confusion_matrix_dt_clustring[1,1],
        confusion_matrix_dt_clustring_opt[1,1],
        confusion_matrix_xgboost_clustring[1,1],
        confusion_matrix_xgb_clustring_opt[1,1],
    ]
})
evaluation[["Precision", "Recall", "F1-score", "ROC-AUC"]] = evaluation[["Precision", "Recall", "F1-score", "ROC-AUC"]].round(3)


evaluation.to_csv(
    "../data/model_evaluation/model_evaluation.csv",
    mode="a",
    index=False,
    header = False
)

In [31]:
# X_test = df.iloc[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]]  # Example indices for test data

# y_true = df.iloc[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]]["Churn"]


# X_test_processed = prepocessor.transform(X_test)

# proba = best_model_dt.predict_proba(X_test_processed)[:, 1]

# pred = (proba >= 0.5).astype(int)


# y_true = y_true.map({"No": 0, "Yes": 1})

# print("Réel       :", y_true.values)
# print("Prédit     :", pred)